# GEOG 214A — Lab 1: VS Code and NASA Earthdata Access
**Name:** Houpu Li
**Course:** GEOG 214A, Fall 2026

**AI usage statement.** I used Claude (Anthropic) to help draft the code and written answers in this notebook.
I checked every numeric answer by running the code cells myself, compared the HLS values against the
printed file metadata, and edited the written explanations where needed.

## Part 0. Environment Setup

In [ ]:
# Sanity check for Part 0 -- confirms Python is on your PATH and that the selected kernel can
# run shell commands. If this errors, revisit the corresponding step above before continuing.
!python3 --version

## Part 1. Python Crash Course
### 1.1 Numbers, Strings, and Booleans

In [ ]:
dn = 214                     # a raw digital number -- int
reflectance = 0.342          # a physical, scaled value -- float
band_name = "B04"            # a band identifier -- str
is_saturated = dn >= 255     # a yes/no question -- bool

print(type(dn), type(reflectance), type(band_name), type(is_saturated))

### 1.2 Variables, Comments, and f-strings

In [ ]:
bit_depth = 12                    # HLS/EMIT-class sensors are commonly 12-bit
max_dn = 2 ** bit_depth - 1        # ** is exponentiation; a 12-bit sensor tops out at 4095

print(f"A {bit_depth}-bit sensor saturates at a DN of {max_dn}.")

#### Required (8 pts) — NDVI for two pixels

In [ ]:
# Pixel 1
red, nir = 0.08, 0.45
ndvi_1 = (nir - red) / (nir + red)

# Pixel 2
red, nir = 0.25, 0.28
ndvi_2 = (nir - red) / (nir + red)

print(f"Pixel 1 NDVI = {ndvi_1:.4f}")
print(f"Pixel 2 NDVI = {ndvi_2:.4f}")

**Answer.** Pixel 1 NDVI ≈ **0.698**; Pixel 2 NDVI ≈ **0.057**.
Pixel 1 reflects much more NIR than red, which is typical of dense, healthy vegetation.
Pixel 2 has nearly equal red and NIR reflectance, so NDVI is close to 0, which is typical of bare soil,
pavement, or very sparse vegetation.

### 1.3 Lists

In [ ]:
hls_bands = ["B02", "B03", "B04", "B05"]   # Blue, Green, Red, NIR -- the bands Part 3 downloads

print(hls_bands[0])       # first band  -> 'B02' (Blue)
print(hls_bands[-1])      # last band   -> 'B05' (NIR)
print(hls_bands[1:3])     # Green through Red (index 3, NIR, is excluded)
print(len(hls_bands))     # how many bands

### 1.4 Tuples

In [ ]:
sb_bbox = (-119.90, 34.35, -119.60, 34.50)   # (west, south, east, north) -- same bbox as Session Setup below
west, south, east, north = sb_bbox            # unpacking

print(f"Study area spans {east - west:.2f} degrees of longitude and {north - south:.2f} of latitude.")

### 1.5 Dictionaries

In [ ]:
# A stand-in for the kind of nested metadata a real `earthaccess` granule hands you in Part 3
granule_metadata = {
    "ShortName": "HLSL30",
    "CloudCover": 12.4,
    "TemporalExtent": {"BeginningDateTime": "2023-06-01T18:32:11Z"},
}
print(granule_metadata["ShortName"])
print(granule_metadata["TemporalExtent"]["BeginningDateTime"])

# A dictionary of named field sites -- name -> (lon, lat)
study_sites = {
    "Ocean": (-119.7988, 34.3907),
    "Vegetation": (-119.85, 34.44),
    "Urban": (-119.70, 34.42),
}
print(study_sites["Vegetation"])

#### Required (8 pts) — NDVI and NDWI from a dictionary

In [ ]:
bands = {"red": 0.08, "green": 0.10, "nir": 0.45}

ndvi = (bands["nir"] - bands["red"]) / (bands["nir"] + bands["red"])
ndwi = (bands["green"] - bands["nir"]) / (bands["green"] + bands["nir"])

print(f"NDVI = {ndvi:.4f}")
print(f"NDWI = {ndwi:.4f}")

**Answer.** NDVI ≈ **0.698** and NDWI ≈ **−0.636**. The high positive NDVI and strongly negative NDWI
(NIR far above green) both indicate a vegetated pixel, not open water.

### 1.6 Comparisons and Booleans

In [ ]:
cloud_cover = 12.4
is_usable = cloud_cover < 20
print("Usable scene?", is_usable)

dn = 255
is_saturated = dn >= (2 ** 8 - 1)   # true ceiling of an 8-bit sensor
print("Saturated pixel?", is_saturated)

### 1.7 for Loops and List Comprehensions

In [ ]:
for band in hls_bands:
    print("Band:", band)

for site, (lon, lat) in study_sites.items():
    print(f"{site}: lon={lon}, lat={lat}")

# List comprehension -- pull the band number out of each name in one line
band_numbers = [int(b[1:]) for b in hls_bands]
print(band_numbers)

### 1.8 Functions

In [ ]:
def apply_scale_factor(dn, scale_factor=0.0001, fill_value=-9999):
    """Convert a raw digital number to a physical unit (e.g., reflectance)."""
    if dn == fill_value:
        return None   # a documented "no data" sentinel, not a real measurement
    return dn * scale_factor

print(apply_scale_factor(3420))
print(apply_scale_factor(-9999))

### 1.9 NumPy Arrays

In [ ]:
import numpy as np

# A tiny stand-in for one corner of a real HLS band: a 4x4 grid of scaled-integer DNs
dn_array = np.array([
    [3120, 3180, 3050, 4200],
    [3090, 3110, 6200, -9999],  # -9999 here is a documented cloud fill value, not a real measurement
    [3200, 3250, 3300, 3280],
    [3000, 3050, 3100, 3150],
])

print("Shape:", dn_array.shape)     # (rows, cols)
print("Data type:", dn_array.dtype)

scale_factor = 0.0001               # same value you'll pull from the real HLS User Guide in Part 3
reflectance = dn_array * scale_factor   # vectorized -- every element scaled in one line, no loop
print(reflectance)

# Boolean masking -- flag anything above a physically sane reflectance ceiling
suspect = reflectance > 0.6
print("Suspect pixels:\n", suspect)

#### Required (8 pts) — Find the fill value with a boolean mask

In [ ]:
fill_mask = dn_array == -9999

print("Fill mask:\n", fill_mask)
print("Fill value(s):", dn_array[fill_mask])
print("Location(s) (row, col):", [(int(r), int(c)) for r, c in zip(*np.where(fill_mask))])
print("Number of fill pixels:", fill_mask.sum())

# Mask the fill value before scaling so it does not become a fake reflectance of -0.9999
reflectance_clean = np.where(fill_mask, np.nan, dn_array * scale_factor)
print("Valid reflectance range:", np.nanmin(reflectance_clean), "-", np.nanmax(reflectance_clean))

**Answer.** The mask picks out one value, **−9999**, at row 1, column 3. If it is not masked, scaling turns it
into a reflectance of −0.9999, which is physically impossible. The 6200 DN (reflectance 0.62) is flagged as
"suspect" by the earlier mask, but it is a real measurement, not a fill value.

## Part 2. Core Concepts

In [ ]:
rows, cols, bands = 300, 200, 100   # rows ("lat"), cols ("lon"), band

# Index as (column, row, band), 0-based, so the first band of the first pixel is (0, 0, 0).

#### Required (7 pts) — Counting pixel-band elements

In [ ]:
n_pixels = rows * cols
n_elements = rows * cols * bands
print(f"Pixels (spatial locations): {n_pixels:,}")
print(f"Pixel-band elements:        {n_elements:,}")

**Answer.** 300 × 200 × 100 = **6,000,000** pixel-band elements.
A pixel is one spatial location (one row/column cell); there are 300 × 200 = 60,000 pixels.
Each pixel holds one value per band, so each pixel carries 100 values. A pixel-band element is one stored
number: the value of one band at one pixel. That is why the count is pixels × bands, not just pixels.

#### Required (7 pts) — Choosing a data type, scale, and offset

In [ ]:
# Reflectance 0-1 at 0.01 precision -> 101 distinct values (0.00, 0.01, ..., 1.00)
n_levels = int(round(1 / 0.01)) + 1
print("Distinct values needed:", n_levels)

for name, nbits in [("uint8", 8), ("uint16", 16)]:
    print(f"{name}: range 0-{2**nbits - 1}, fits 101 levels? {2**nbits >= n_levels}")

# Storage with uint8
bits_per_value = 8
total_bits = rows * cols * bands * bits_per_value
print(f"Total bits (uint8): {total_bits:,}  (= {total_bits/8:,.0f} bytes = {total_bits/8/1e6:.1f} MB)")

# Check a proposed (S, O) by encoding both ends of the physical range
def encode(r, S, O):
    return r * S + O

for S, O in [(100, 0), (500, 0)]:
    lo, hi = encode(0.0, S, O), encode(1.0, S, O)
    print(f"S={S}, O={O}: DN range {lo:.0f}-{hi:.0f}, fits uint8 (0-255)? {0 <= lo and hi <= 255}")

**Answers.**
1. **Smallest data type: `uint8`.** At 0.01 precision, reflectance 0–1 has 101 distinct values. `uint8` stores
   0–255 (256 values), which is enough. A 7-bit type would also fit, but standard data types come in 8-bit steps.
2. **Storage:** 6,000,000 elements × 8 bits = **48,000,000 bits** (6,000,000 bytes, 6 MB).
3. **Scale:** **S = 100**, so DN = 100 × R. For example, 0.52 → 52, and 1.00 → 100. Decoding is R = DN / 100.
4. **Offset:** **O = 0** is enough. Reflectance is never negative in the stated range (0–1), so no value needs
   to be shifted up to fit an unsigned type. A nonzero offset would only be needed if the product allowed
   negative values (real surface-reflectance products can produce small negative values from atmospheric
   over-correction), or if we wanted to reserve DN 0 as a fill value.
5. **S = 500, O = 0 does not work with `uint8`.** R = 1.0 gives DN = 500, which is above the uint8 maximum
   of 255. Any R above 0.51 (255/500) would overflow or wrap around, silently corrupting the bright end
   (vegetation in NIR, snow, clouds). S = 500 would work only with `uint16` (max 65,535), at twice the storage.

### Byte order and band interleave

In [ ]:
bytes_per_value = 2  # 16-bit unsigned integer = 2 bytes per stored value

#### Required (7 pts) — Storage size and interleave

In [ ]:
total_bytes = rows * cols * bands * bytes_per_value
print(f"Total size: {total_bytes:,} bytes")
print(f"            = {total_bytes / 1e6:.2f} MB (decimal, 10^6 bytes)")
print(f"            = {total_bytes / 2**20:.2f} MiB (binary, 2^20 bytes)")

**Answer.** 300 × 200 × 100 × 2 = **12,000,000 bytes = 12 MB** (≈ 11.44 MiB).
**The size does not depend on interleave.** BSQ, BIL, and BIP store exactly the same 6,000,000 values; they
only change the *order* in which the values are written (band by band, line by line, or pixel by pixel). The
interleave affects how fast certain read patterns are (e.g., BIP is fast for reading a full spectrum at one
pixel; BSQ is fast for reading one whole band), but not the total number of bytes, assuming no compression.

## Part 3. Finding and Opening Data with `earthaccess`
### Session Setup

In [ ]:
%pip install -q earthaccess rasterio numpy matplotlib

In [ ]:
# Quick environment check -- run this first.
import importlib.util

required = ["earthaccess", "rasterio", "numpy", "matplotlib"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print("Missing packages:", missing)
    print("Install them with:  pip install " + " ".join(missing))
else:
    print("All required packages are importable. You're ready to go.")

In [ ]:
import earthaccess

# First run will prompt for your NASA Earthdata Login username/password and cache it in ~/.netrc
auth = earthaccess.login()

In [ ]:
# Santa Barbara, CA study area: (west, south, east, north) in decimal degrees
sb_bbox = (-119.90, 34.35, -119.60, 34.50)

# A shared date range to search within -- adjust to taste. Using a wide range increases
# the odds of finding a cloud-free scene for every collection.
date_range = ("2023-01-01", "2023-12-31")

download_dir = "./data"

### Searching for Data
**1. By collection `short_name`:**

In [ ]:
results = earthaccess.search_data(
    short_name="HLSL30",       # Harmonized Landsat Sentinel-2, Landsat-like product
    bounding_box=sb_bbox,
    temporal=date_range,
    count=5,
)
for g in results:
    print(g)

**2. By keyword** (collection search first, then granule search):

In [ ]:
# Step 1: find candidate collections by keyword (this is search_datasets, not search_data)
candidate_collections = earthaccess.search_datasets(
    keyword="MASTER",   # try other keywords too, e.g. "AVIRIS", "hyperspectral", "thermal"
    count=10,
)
for c in candidate_collections:
    print(c["umm"]["ShortName"], "-", c["umm"].get("EntryTitle"))

In [ ]:
# Step 2: once you've picked a collection above, search its granules by short_name --
# this is the same search_data() call pattern used throughout the rest of this notebook.
chosen_short_name = candidate_collections[0]["umm"]["ShortName"]

keyword_results = earthaccess.search_data(
    short_name=chosen_short_name,
    bounding_box=sb_bbox,
    temporal=date_range,
    count=10,
)
for g in keyword_results:
    print(g)

#### Required (7 pts) — Inspect granule metadata before downloading

In [ ]:
import json

g = results[0]

print("SpatialExtent:")
print(json.dumps(g["umm"]["SpatialExtent"], indent=2))

print("\nTemporalExtent:")
print(json.dumps(g["umm"]["TemporalExtent"], indent=2))

size_mb = g.size()
print(f"\nSize: {size_mb:.1f} MB")
print(f"Ratio to the 12 MB hypothetical scene in Part 2: {size_mb / 12:.1f}x")

print("\nData links:")
for link in g.data_links():
    print(" ", link)

**Answer.** The first HLSL30 granule is about **200 MB**, roughly 17× the 12 MB hypothetical scene from Part 2,
even though it is one Landsat scene with only ~15 files (bands + QA/angle layers).

Checking metadata first is useful because it lets you decide *before* spending time, bandwidth, and disk space:
- **SpatialExtent:** confirms the footprint actually covers the study area. A granule can intersect the
  bounding box only at its edge (e.g., tile T10SGD vs. T11SKU both touch Santa Barbara), so you can choose the
  tile that covers the area best and avoid downloading duplicates.
- **TemporalExtent:** confirms the acquisition date and time match the event or season you need.
- **size():** tells you whether the download is feasible on your machine and how long it will take, which
  matters when a search returns dozens or hundreds of granules.
- **data_links():** shows which files are included, so you can download only the bands you need (e.g., B04 and
  B05) instead of the whole granule, and confirms the file format (.tif vs. .hdf5).

**Filtering by cloud cover**

In [ ]:
clear_results = earthaccess.search_data(
    short_name="HLSL30",
    bounding_box=sb_bbox,
    temporal=date_range,
    cloud_cover=(0, 10),   # 0-10% cloud cover
    count=5,
)
print(f"{len(clear_results)} scene(s) under 10% cloud cover")

#### Required (7 pts) — Discussion: `short_name` search vs. keyword workflow

**Direct `short_name` search** is the right choice when I already know the exact product, such as HLSL30,
HLSS30, or EMITL2ARFL. It is one call, it is reproducible, and the same code returns the same product next year.

**The two-step keyword workflow** (`search_datasets` → pick a `ShortName` → `search_data`) is the right choice
when I do not know the exact product name: exploring what exists for a topic ("thermal", "hyperspectral"),
or working with airborne campaign data such as MASTER, where each campaign or deployment gets its own
collection (e.g., `MASTER_FireSense_Spring_2025_2439`, `MASTER_WDTS_Spring_2024_2383`).

**What a stable `short_name` tells you.** A stable, short `short_name` (HLSL30, MOD09GA) signals an
**operational** product: a continuous, systematically processed record with a fixed algorithm and versioning,
designed for long time series. Short names that include a campaign, season, year, or a numeric dataset ID
signal a **campaign-based** product: data collected during a limited flight campaign, over specific sites and
dates, published as separate collections. For campaign data you should expect irregular coverage in space and
time, and you have to rediscover the right collection for each campaign instead of reusing one name.

### Opening What You Found

In [ ]:
demo_files = earthaccess.download(clear_results[:1], f"{download_dir}/demo")
print(demo_files)

In [ ]:
import glob
import numpy as np
import rasterio
import os
band_path = sorted(glob.glob(f"{download_dir}/demo/*B04.tif"))[0]   # red band
print("File size (MB):", os.path.getsize(band_path) / 10**6)

with rasterio.open(band_path) as src:
    print("dtype:", src.dtypes[0])
    print("nodata / fill value:", src.nodata)
    print("shape (band, row, col):", (src.count, src.height, src.width))
    scale_factor = float(src.tags()["scale_factor"])   # read straight from the file, not hardcoded
    print("scale factor (from the file's own tags):", scale_factor)
    raw = src.read(1)

print("\nRaw stored values (min/max):", raw.min(), raw.max())

reflectance = raw.astype(float) * scale_factor
print("Scaled reflectance (min/max):", np.nanmin(reflectance), np.nanmax(reflectance))

#### Required (7 pts) — Expected vs. actual file size

In [ ]:
with rasterio.open(band_path) as src:
    n_bands, n_rows, n_cols = src.count, src.height, src.width
    bytes_per_value_real = np.dtype(src.dtypes[0]).itemsize
    compression = src.compression
    tiled = src.profile.get("tiled")
    overviews = src.overviews(1)

expected_bytes = n_rows * n_cols * n_bands * bytes_per_value_real
actual_bytes = os.path.getsize(band_path)

print(f"Shape: {n_bands} x {n_rows} x {n_cols}, dtype bytes = {bytes_per_value_real}")
print(f"Expected (uncompressed): {expected_bytes:,} bytes = {expected_bytes / 1e6:.2f} MB")
print(f"Actual on disk:          {actual_bytes:,} bytes = {actual_bytes / 1e6:.2f} MB")
print(f"Actual / expected:       {actual_bytes / expected_bytes:.2f}")
print("Compression:", compression, "| tiled:", tiled, "| overviews:", overviews)

**Answer.** The band is `int16` (2 bytes per value) with shape (1, 3660, 3660), so the expected size is
1 × 3660 × 3660 × 2 = **26,791,200 bytes ≈ 26.79 MB** (≈ 25.55 MiB).

The actual file on disk is **smaller** (see the printed ratio above), so the two do not match. The reason is
**compression**: HLS bands are Cloud-Optimized GeoTIFFs stored with lossless compression (DEFLATE), so repeated
and similar values (ocean, fill areas, smooth surfaces) take far less than 2 bytes each. The file also contains
small extras that the simple formula ignores — GeoTIFF header and tags, internal tile index, and reduced-
resolution overviews — but these add only a little compared to what compression saves. The rows × cols × bands
× bytes formula gives the **in-memory** size after reading, not the on-disk size of a compressed file.

#### Required (7 pts) — Discussion: forgetting the scale factor

Without the scale factor, the "reflectance" values would be the raw integers, roughly 0 to ~10,000
(plus −9999 fill), instead of 0 to ~1. In a single band this mistake is fairly **obvious** if you look at the
numbers, because reflectance above 1 is physically impossible.

However, it can easily be **silent and plausible-looking**:
- **Maps look the same.** A color ramp stretches to the data range, so an unscaled band produces exactly the
  same picture as a scaled one. Nothing looks wrong unless you read the legend.
- **Ratio indices hide it.** NDVI = (NIR − Red)/(NIR + Red) gives the same result with or without the scale,
  because the factor cancels (as long as offset is 0). This hides the error until you do something that is
  not scale-invariant.
- **Downstream steps break quietly.** Fixed thresholds (e.g., "reflectance > 0.3"), physical models, unit
  conversions, comparisons with other sensors or field spectra, and band math with an additive term
  (e.g., EVI or SAVI with L = 0.5) would all give wrong but numerically reasonable outputs.
- **Fill values make it worse.** If −9999 is not masked, it gets scaled or averaged into results.

So the error is obvious if you inspect the values, but it can produce a plausible map and wrong analysis if you
only look at images or scale-invariant indices.

**Band interleave check — rasterio always returns (band, row, col)**

In [ ]:
all_band_paths = sorted(glob.glob(f"{download_dir}/demo/*B0[2-5].tif"))
with rasterio.open(all_band_paths[0]) as ref:
    stack = np.stack([rasterio.open(p).read(1) for p in all_band_paths])

print("Stacked array shape (band, row, col):", stack.shape)

#### Required (7 pts) — Try it: skip the download (stream with `earthaccess.open()`)

In [ ]:
try:
    streamed_files = earthaccess.open(clear_results[:1])
    print(f"{len(streamed_files)} file-like object(s) opened without downloading:")
    for sf in streamed_files:
        print(" ", sf)
    with rasterio.open(streamed_files[0]) as src:
        print("\nFirst file -- dtype:", src.dtypes[0], " shape:", (src.count, src.height, src.width))
except Exception as e:
    print(f"Streaming didn't work in this environment ({e}) -- that's fine, .download() above already has the data.")

**Result.** `earthaccess.open()` returns one file-like object per file in the granule (15 for an HLSL30 granule:
bands B01–B11, Fmask, and the four sun/view angle layers). `rasterio` opens the first object directly from the
HTTPS link, and it reports the same `int16` dtype and (1, 3660, 3660) shape as the downloaded file, without
writing anything to disk. Streaming is convenient for a quick look or for reading only a subset of a COG, but
each read goes over the network, so a local download is faster for repeated analysis.